# 22 — Busca Ampla de Hiperparâmetro CatBoost (2ª rodada)
## PRT Seguros

O notebook `11` só tunou `depth`, `learning_rate`, `l2_leaf_reg`, `random_strength`. Aqui ampliamos
pra parâmetros que controlam a amostragem interna de cada árvore (`bootstrap_type`,
`bagging_temperature`, `subsample`), a quantização das variáveis contínuas (`border_count`) e o
modo de boosting (`boosting_type`) — nunca testados antes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
X_train, y_train = train[feature_cols], train[TARGET_COL]
X_val, y_val = val[feature_cols], val[TARGET_COL]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

param_dist = {
    "depth": [4, 5, 6, 7, 8, 9, 10],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [1, 3, 5, 7, 9, 12],
    "border_count": [32, 64, 128, 254],
    "bootstrap_type": ["Bayesian", "Bernoulli", "MVS"],
    "boosting_type": ["Plain", "Ordered"],
}

busca = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        iterations=300, scale_pos_weight=neg_pos_ratio,
        eval_metric="AUC", random_seed=RANDOM_STATE, verbose=False,
    ),
    param_distributions=param_dist,
    n_iter=30, cv=3, scoring="roc_auc",
    random_state=RANDOM_STATE, n_jobs=3, verbose=1, error_score=np.nan,
)
busca.fit(X_train, y_train)
print(f"Melhores parâmetros: {busca.best_params_}")
print(f"Melhor AUC-ROC (CV, 300 iterations): {busca.best_score_:.4f}")


## Retreinar com early stopping usando os melhores parâmetros

In [ ]:
X_tr_es, X_es, y_tr_es, y_es = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

BONUS_PARAMS_V2 = dict(busca.best_params_)
if BONUS_PARAMS_V2.get("bootstrap_type") not in ("Bayesian",):
    BONUS_PARAMS_V2.pop("bagging_temperature", None)

modelo_v2 = CatBoostClassifier(
    iterations=3000, **BONUS_PARAMS_V2, scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=200,
)
modelo_v2.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))
print(f"Melhor iteração: {modelo_v2.get_best_iteration()}")

proba_val = modelo_v2.predict_proba(X_val)[:, 1]
auc_v2 = roc_auc_score(y_val, proba_val)
print(f"\nAUC-ROC val (novos parâmetros): {auc_v2:.4f}")
print(f"Referência (params antigos, notebook 11): 0.8263")

import json
with open("dados_processados/melhores_params_cat_v2.json", "w") as f:
    json.dump(BONUS_PARAMS_V2, f, indent=2)
print("Parâmetros salvos em dados_processados/melhores_params_cat_v2.json")
